In [ ]:
#загрузка библиотек
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as statf
import statsmodels.stats.diagnostic as dg
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.sandwich_covariance import cov_cluster_2groups
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
dta = pd.read_parquet("lag_model_monthly_dyads_giga_r20_50k_100k.parquet")
dta

In [ ]:
#посмотрим, что все верно загрузилось
print("число каналов:", len(set(dta["channel_i"]) | set(dta["channel_j"])))
print("число диад:", dta["dyad_id"].nunique())
print("число месяцев:", dta["month"].nunique())
print("период:", dta["month"].min(), "-", dta["month"].max())

display(dta.isna().mean().sort_values(ascending=False).head(10))

In [ ]:
#добавим fe на кварталы
dta = dta.copy()
dta["month_date"] = pd.to_datetime(dta["month"].astype(str) + "-01")
dta["quarter"] = dta["month_date"].dt.to_period("Q").astype(str)

dta["channel_i_code"] = pd.Categorical(dta["channel_i"]).codes
dta["channel_j_code"] = pd.Categorical(dta["channel_j"]).codes

display(dta[["month", "quarter"]].drop_duplicates().sort_values("month"))

In [ ]:
#визуализируем динамику зависимой переменной
monthly_diag = (dta.groupby("month", observed=True).agg(
        n_dyads=("dyad_id", "size"),
        tie_strength_mean=("tie_strength", "mean"),
        tie_strength_sd=("tie_strength", "std"),
        tie_strength_lag1_mean=("tie_strength_lag1", "mean"),
        posts_sum_mean=("posts_sum", "mean"),).reset_index())
monthly_diag["month_date"] = pd.to_datetime(monthly_diag["month"].astype(str) + "-01")

plt.figure(figsize=(12, 5))
plt.plot(monthly_diag["month_date"], monthly_diag["tie_strength_mean"], marker="o", c='hotpink')
plt.title("Средняя сила тематико-сентиментной связи по месяцам")
plt.xlabel("Месяц")
plt.ylabel("Средняя tie_strength")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#для более детальной визуализации посчитаем месячную корреляцию текущей и лагированной связи
monthly_corr = (dta.dropna(subset=["tie_strength", "tie_strength_lag1"]).groupby("month", observed=True).apply(lambda x: x["tie_strength"].corr(x["tie_strength_lag1"]))
    .reset_index(name="corr_tie_strength_lag1"))

monthly_corr["month_date"] = pd.to_datetime(monthly_corr["month"].astype(str) + "-01")

In [ ]:
#визуализируем месячную корреляцию текущей и лагированной связи
plt.figure(figsize=(12, 5))
plt.plot(monthly_corr["month_date"], monthly_corr["corr_tie_strength_lag1"], marker="o", c='hotpink')
plt.title("Корреляция текущей и лагированной силы связи по месяцам")
plt.xlabel("Месяц")
plt.ylabel("cor(tie_strength_t, tie_strength_t-1)")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Можно сказать, что между силой связи диад в текущем и предыдущем месяце наблюдается устойчивая положительная корреляция. Это свидетельствует о том, что тематико-сентиментная структура сети обладает определенной инерцией: пары каналов, близкие друг к другу в одном месяце, как правило, остаются относительно близкими и в следующем. При этлом  величина корреляции  варьируется во времени, это указывает на наличие периодов большей и меньшей устойчивости сети.

In [ ]:
#распределение зависимой переменной по месяцам
months = sorted(dta["month"].unique())
box_data = [dta.loc[dta["month"] == m, "tie_strength"] for m in months]

plt.figure(figsize=(14, 6))
plt.boxplot(box_data, labels=months, showfliers=False)
plt.title("Распределение силы связи по месяцам")
plt.xlabel("Месяц")
plt.ylabel("tie_strength")
plt.xticks(rotation=90)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
#scatter plot текущего и прошлого значения
plot_sample = dta.sample(min(40000, len(dta)), random_state=42)

plt.figure(figsize=(7, 6))
plt.scatter(plot_sample["tie_strength_lag1"], plot_sample["tie_strength"], alpha=0.15, color='hotpink')
plt.title("Связь между tie_strength_t и tie_strength_t-1")
plt.xlabel("tie_strength_t-1")
plt.ylabel("tie_strength_t")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#стационарность месячного среднего
adf_data = monthly_diag["tie_strength_mean"].dropna()
result = adfuller(adf_data)

print("Statistic:", result[0])
print("p-value:", result[1])

Данные стационарные, дополнительные преобразования временного ряда не требуются.

In [ ]:
#acf и pacf месячного среднего
max_lags = min(8, len(monthly_diag) // 2 - 1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(monthly_diag["tie_strength_mean"], lags=max_lags, ax=ax1, color='hotpink')
ax1.set_title("ACF месячного среднего tie_strength")

plot_pacf(monthly_diag["tie_strength_mean"], lags=max_lags, ax=ax2, method="ywm", color='hotpink')
ax2.set_title("PACF месячного среднего tie_strength")

plt.tight_layout()
plt.show()

ACF и PACF для месячного среднего значения tie_strength не показывают выраженной автокорреляции агрегированного ряда. Но это не противоречит использованию лагированной модели, поскольку основная гипотеза проверяется на уровне диад, а не на уровне месячного среднего.

In [ ]:
#тест бреуша-годфри для агрегированного ряда
monthly_diag = monthly_diag.sort_values("month").copy()
monthly_diag["tie_strength_mean_lag1"] = monthly_diag["tie_strength_mean"].shift(1)

bg_base = monthly_diag.dropna(subset=["tie_strength_mean", "tie_strength_mean_lag1"])
bg_model = statf.ols("tie_strength_mean ~ tie_strength_mean_lag1", data=bg_base).fit()

bg_result = dg.acorr_breusch_godfrey(bg_model, nlags=min(3, len(bg_base) // 3))

print("LM statistic:", bg_result[0])
print("LM p-value:", bg_result[1])
print("F statistic:", bg_result[2])
print("F p-value:", bg_result[3])

In [ ]:
#функции для моделей с кластеризацией по двум каналам
def fit_ols_cluster2(formula, data):
    model = statf.ols(formula, data=data).fit()
    clustered_model = model.get_robustcov_results(
        cov_type="cluster",
        groups=data[["channel_i_code", "channel_j_code"]].to_numpy())
    return model, clustered_model

#функция для таблицы коэффициентов
def coef_table(model, model_name):
    out = pd.DataFrame({
        "model": model_name,
        "term": model.model.exog_names,
        "coef": model.params,
        "se": model.bse,
        "t": model.tvalues,
        "p_value": model.pvalues})

    out["stars"] = np.select(
        [out["p_value"] < 0.001, out["p_value"] < 0.01, out["p_value"] < 0.05, out["p_value"] < 0.1],
        ["***", "**", "*", "."], default="")

    return out

def model_info(model, model_name):
    return pd.DataFrame({
        "model": [model_name],
        "nobs": [int(model.nobs)],
        "r2": [model.rsquared],
        "aic": [model.aic],
        "bic": [model.bic]})

In [ ]:
#кодируем каналы для двухуровневой кластеризации
dta["channel_i_code"] = dta["channel_i"].astype("category").cat.codes
dta["channel_j_code"] = dta["channel_j"].astype("category").cat.codes

In [ ]:
#модель 1: базовая спецификация с лагом
formula_m1 = "tie_strength ~ tie_strength_lag1"

m1, cov_m1 = fit_ols_cluster2(formula_m1, dta)
print(cov_m1.summary())

In [ ]:
#модель 2: лаговая спецификация с квартальными fe
formula_m2 = "tie_strength ~ tie_strength_lag1 + C(quarter)"

m2, cov_m2 = fit_ols_cluster2(formula_m2, dta)
print(cov_m2.summary())

In [ ]:
#модель 3: лаговая спецификация с контролями и квартальными fe
formula_m3 = "tie_strength ~ tie_strength_lag1 + posts_sum + posts_diff + topic_diversity_mean + negative_share_mean + n_common_topics + C(quarter)"

m3, cov_m3 = fit_ols_cluster2(formula_m3, dta)
print(cov_m3.summary())

In [ ]:
#проверяем корреляции между предикторами
vif_vars = [
    "tie_strength_lag1",
    "posts_sum",
    "posts_diff",
    "topic_diversity_mean",
    "negative_share_mean",
    "n_common_topics"
]

dta[vif_vars].corr()

In [ ]:
#считаем vif для предикторов
vif_data = dta[vif_vars].dropna().copy()
vif_data = sm.add_constant(vif_data)

vif_table = pd.DataFrame({"variable": vif_data.columns, "vif": [variance_inflation_factor(vif_data.values, i) for i in range(vif_data.shape[1])]})
vif_table.loc[vif_table["variable"] != "const"].sort_values("vif", ascending=False)

Дополнительно проверили на мультиколлинеарность, оказалось, что posts_sum и posts_diff  сильно связаны между собой. Значит, обе переменные во многом описывают один и тот же аспект активности пары каналов. Поэтому дополнительно оценим компактную спецификацию без posts_diff.

In [ ]:
#модель 3b: спецификация без posts_diff
formula_m3b = "tie_strength ~ tie_strength_lag1 + posts_sum + topic_diversity_mean + negative_share_mean + n_common_topics + C(quarter)"

m3b, cov_m3b = fit_ols_cluster2(formula_m3b, dta)
print(cov_m3b.summary())

In [ ]:
#проверим vif для спецификации без posts_diff
vif_vars_3b = ["tie_strength_lag1", "posts_sum", "topic_diversity_mean", "negative_share_mean", "n_common_topics"]
vif_data_3b = dta[vif_vars_3b].dropna().copy()
vif_data_3b = sm.add_constant(vif_data_3b)

vif_table_3b = pd.DataFrame({"variable": vif_data_3b.columns, "vif": [variance_inflation_factor(vif_data_3b.values, i) for i in range(vif_data_3b.shape[1])]})
vif_table_3b.loc[vif_table_3b["variable"] != "const"].sort_values("vif", ascending=False)

После исключения posts_diff мультиколлинеарность снизилась: VIF для posts_sum стал низким, проблемными остались только topic_diversity_mean и n_common_topics, но их значения умеренно высокие. При этом основной коэффициент tie_strength_lag1 почти не изменился, тоесть вывод об устойчивости тематико-сентиментных связей не зависит от удаления posts_diff.

Поэтому выбираем 3b в качестве финальгной модели. Она включает лаг зависимой переменной, квартальные фиксированные эффекты и основные контрольные переменные, но исключает posts_diff, чтобы минимизировать проблему мультиколлинеарности.

In [ ]:
#выведем основные коэффициенты
model_tables = pd.concat([
    coef_table(cov_m1, "m1_base"),
    coef_table(cov_m2, "m2_quarter_fe"),
    coef_table(cov_m3, "m3_controls_quarter_fe"),
    coef_table(cov_m3b, "m3b_final")], ignore_index=True)

model_infos = pd.concat([
    model_info(m1, "m1_base"),
    model_info(m2, "m2_quarter_fe"),
    model_info(m3, "m3_controls_quarter_fe"),
    model_info(m3b, "m3b_final")], ignore_index=True)

main_terms = [
    "Intercept",
    "tie_strength_lag1",
    "posts_sum",
    "posts_diff",
    "topic_diversity_mean",
    "negative_share_mean",
    "n_common_topics"]

main_results = model_tables.loc[model_tables["term"].isin(main_terms)].copy()

display(main_results)
display(model_infos)

In [ ]:
#остатки финальной модели во времени
dta_model = dta.loc[m3b.model.data.row_labels].copy()
dta_model["resid_m3b"] = m3b.resid

resid_by_month = (dta_model.groupby("month", observed=True)["resid_m3b"].mean().reset_index())

resid_by_month["month_date"] = pd.to_datetime(resid_by_month["month"].astype(str) + "-01")

plt.figure(figsize=(12, 5))
plt.plot(resid_by_month["month_date"], resid_by_month["resid_m3b"], marker="o", color='hotpink')
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Средние остатки финальной модели по месяцам")
plt.xlabel("Месяц")
plt.ylabel("Средний остаток")
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
#acf и pacf остатков финальной модели
max_lags = min(8, len(resid_by_month) // 2 - 1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

plot_acf(resid_by_month["resid_m3b"], lags=max_lags, ax=ax1, color='hotpink')
ax1.set_title("ACF средних месячных остатков финальной модели")

plot_pacf(resid_by_month["resid_m3b"], lags=max_lags, ax=ax2, method="ywm", color='hotpink')
ax2.set_title("PACF средних месячных остатков финальной модели")

plt.tight_layout()
plt.show()

In [ ]:
#тест бреуша-годфри для средних месячных остатков
resid_bg = resid_by_month.copy()
resid_bg["resid_lag1"] = resid_bg["resid_m3b"].shift(1)
resid_bg = resid_bg.dropna(subset=["resid_m3b", "resid_lag1"])

resid_bg_model = statf.ols("resid_m3b ~ resid_lag1", data=resid_bg).fit()
resid_bg_result = dg.acorr_breusch_godfrey(resid_bg_model, nlags=min(3, len(resid_bg) // 3))

print("LM statistic:", resid_bg_result[0])
print("LM p-value:", resid_bg_result[1])
print("F statistic:", resid_bg_result[2])
print("F p-value:", resid_bg_result[3])